# Sizing the sc_int2 Integrator

Adapted from `macros/sw_tgate_cmfb2/scripts/sizing/size_sw_tgate_cmfb2.ipynb`, which in turn follows `macros/ota_fcasc_fd/scripts/sizing/lookup_commands.ipynb`. The system numbers (OSR, kT/C, C1, C2, settling budget) are the system-level ones: 1.5 V supply, 1 mV LSB, OSR 512 at 1.024 MHz, first-order CIFB loop with a1 = 1/3, C1 = 64 fF.

**Copyright 2026 Michael Koefinger**

SPDX-License-Identifier: Apache-2.0 WITH SHL-2.1

`sc_int2` was sized by hand and by simulation. This notebook derives the sizes from the spec, then reads the drawn sizes out of the schematic and compares them. It changes nothing. Where the two disagree the table says so.

Every number here is a model estimate at tt, 300 K. Simulated values come from `macros/sc_int2/README.md` and are sanity checks of one or a few runs, not a characterisation.

Nothing in this block is biased by a gm/ID choice except the OTA, and the OTA has its own script, `macros/ota_fcasc2/scripts/sizing/size_ota_fcasc2.py`. It is run here and its gm, beta, tau, slew current and output range are taken as inputs. The switch is `sw_tgate`, sized in `macros/sw_tgate/scripts/sizing/size_sw_tgate.ipynb`, and here only its Ron is evaluated again for the paths of this integrator. What is derived in this notebook: the capacitors from kT/C and the coefficient, the swing, the settling and slewing against the two phases, the switch Ron against the same budget, the leak of the virtual ground, and what limits the recovery after `di_rst_b`.

In [1]:
# Packages
from pygmid import Lookup as lk
import numpy as np
import re
import io
import runpy
import warnings
import contextlib
from pathlib import Path

In [2]:
# Spec: system level, then where things live
VDD     = 1.5
FS      = 1.024e6               # sampling clock, Hz
FB      = 1e3                   # signal band edge, Hz
OSR     = FS / (2 * FB)         # 512
V_REF   = VDD / 2               # full scale 0.75 V around vdd/2, the DAC levels are vdd and vss
V_LSB   = 1e-3
TEMP, KB = 300, 1.38e-23
A1      = 1 / 3                 # 2*C1/C2, the loop coefficient of the system-level design
C_FLOOR = 10e-15                # the course rule, no capacitor below 10 fF
A0_INT_DB = 50.0                # integrator dc gain wanted by the system simulation
V_SAT   = 1.41                  # differential OTA output limit seen in reset, README of dsm2
CAP_MODEL = 1.287e-15           # F/um2, cap_cmomf area law on Metal1 to Metal4
C_UNIT_REAL = 32e-15            # estimated real value of the 3.0u x 11.39u unit, README
TS2     = 1 / (2 * FS)          # one phase
CELL    = 'sc_int2'
DATA    = Path('../../../macros/inverter/scripts/sizing/data')
SCH     = Path('../schematic') / (CELL + '.sch')
SCH_TG  = Path('../schematic/sw_tgate.sch')
OTA_PY  = Path('size_ota_fcasc2.py')

### The OTA numbers, taken from its own script

`size_ota_fcasc2.py` is run as a module without its printout, which gives the unit-device design point it prints as `equal_l`. That is the point drawn in `ota_fcasc2`. Its caps are the real values of the class design (C1 64 fF, C2 384 fF, fanout 2), the model caps of the schematic are handled further down.

In [3]:
warnings.filterwarnings('ignore')
with contextlib.redirect_stdout(io.StringIO()):
    OTA = runpy.run_path(str(OTA_PY))
OTA_PT = OTA['evaluate'](OTA['POINTS']['equal_l'])
GM1, BETA_OTA, TAU_OTA = OTA_PT['gm1'], OTA_PT['beta'], OTA_PT['tau']
A0_OTA, SWING_OTA, CGG1 = OTA_PT['a0'], OTA_PT['swing'], OTA_PT['cgg1']
I_SLEW = OTA['id1']             # the folded cascode delivers at most Id1 into the output in either direction
FANOUT = OTA['fanout']
print(f"OTA (model): gm1 {GM1*1e6:.2f} uS, beta {BETA_OTA:.4f}, tau {TAU_OTA*1e9:.2f} ns, A0 {20*np.log10(A0_OTA):.1f} dB")
print(f"             output range {SWING_OTA:.3f} V differential, slew current Id1 {I_SLEW*1e9:.0f} nA, Cgg1 {CGG1*1e15:.1f} fF")

OTA (model): gm1 15.15 uS, beta 0.7861, tau 17.65 ns, A0 55.3 dB
             output range 1.604 V differential, slew current Id1 600 nA, Cgg1 40.5 fF


### Capacitors from kT/C and the coefficient

The class formula puts the thermal noise of the sampled input at the same level as the quantization noise of one LSB, `C1 >= kT / (OSR * vn^2)`. The circuit gives each of its four sampling events a kT/C1 and the gain `2*C1/C2` refers them back to the input as `kT/C1` per sample, so the in-band noise is `sqrt(kT / (C1 * OSR))`.

The noise floor is far below anything drawn here. C1 is set by the 10 fF rule, by leakage and by droop (a half size C1 costs 1.7 times the leak and 4.5 times the droop, `doc/noise-budget.md`), not by kT/C. C2 follows from the coefficient.

In [4]:
vn      = V_LSB / np.sqrt(12)
c1_min  = KB * TEMP / (OSR * vn**2)
C1      = 64e-15                # the system-level choice, above the kT/C minimum
C2      = 2 * C1 / A1
noise_c1 = np.sqrt(KB * TEMP / (C1 * OSR))
print(f"OSR {OSR:.0f}, noise budget {vn*1e6:.1f} uVrms")
print(f"C1 >= {c1_min*1e15:.3f} fF from kT/C, {C_FLOOR*1e15:.0f} fF from the course rule, chosen {C1*1e15:.0f} fF")
print(f"C2 = 2*C1/a1 = {C2*1e15:.0f} fF, C1/C2 = {C1/C2:.4f}")
print(f"kT/C1 in band at {C1*1e15:.0f} fF: {noise_c1*1e6:.2f} uVrms, {20*np.log10(vn/noise_c1):.1f} dB below the budget (sim, ideal switches: 10.61 uVrms)")

OSR 512, noise budget 288.7 uVrms
C1 >= 0.097 fF from kT/C, 10 fF from the course rule, chosen 64 fF
C2 = 2*C1/a1 = 384 fF, C1/C2 = 0.1667
kT/C1 in band at 64 fF: 11.24 uVrms, 28.2 dB below the budget (sim, ideal switches: 10.61 uVrms)


### Read the sizes out of the schematic

The schematic is the source of truth. The capacitors are `cap_cmomf` vector instances, so the count comes from the bus name and the value from the area law of the model. The dummy ring and the decoupling (`CDUM_*`) touch no signal node and are skipped. The transmission gate sizes come from the `sw_tgate` cell, the instances are counted in `sc_int2`.

In [5]:
def read_caps(path):
    txt = path.read_text()
    caps = {}
    for m in re.finditer(r'C \{cap_cmomf\.sym\} \S+ \S+ \S+ \S+ \{name=(\w+)(?:\[(\d+):(\d+)\])?\n(.*?)\n\}', txt, re.S):
        if m.group(1).startswith('CDUM'):
            continue
        a = dict(re.findall(r'^(\w+)=(\S+)$', m.group(4), re.M))
        n = 1 if m.group(2) is None else abs(int(m.group(2)) - int(m.group(3))) + 1
        caps[m.group(1)] = (n * int(a['m']), float(a['w']) * 1e6, float(a['l']) * 1e6)
    return caps

def read_tgate(path):
    txt = path.read_text()
    dev = {}
    for m in re.finditer(r'\{name=(M[NP])\n(.*?)\n\}', txt, re.S):
        a = dict(re.findall(r'^(\w+)=(\S+)$', m.group(2), re.M))
        dev[m.group(1)] = (float(a['w'].rstrip('u')), float(a['l'].rstrip('u')))
    return dev['MN'][0], dev['MP'][0], dev['MN'][1]

CAPS = read_caps(SCH)
WN, WP, L_SW = read_tgate(SCH_TG)
SWITCHES = re.findall(r'C \{sw_tgate\.sym\} \S+ \S+ \S+ \S+ \{name=(\w+)', SCH.read_text())
c_dev = {k: CAP_MODEL * w * l for k, (n, w, l) in CAPS.items()}
for k, (n, w, l) in CAPS.items():
    print(f'{k:6s} {n:2d} x {w:.1f}u x {l:.2f}u, {c_dev[k]*1e15:.2f} fF each by the model')
print(f'{len(SWITCHES)} sw_tgate instances, Wn {WN} um, Wp {WP} um, L {L_SW} um')

C1P     2 x 3.0u x 11.39u, 43.98 fF each by the model
C1N     2 x 3.0u x 11.39u, 43.98 fF each by the model
C2P    12 x 3.0u x 11.39u, 43.98 fF each by the model
C2N    12 x 3.0u x 11.39u, 43.98 fF each by the model
14 sw_tgate instances, Wn 2.0 um, Wp 4.0 um, L 1.0 um


The schematic carries the model value of the unit, 43.98 fF, and the layout README estimates 32 fF for the real one. The ratios do not care, the absolute values do wherever a time constant appears, so both are carried below.

In [6]:
n1, n2 = CAPS['C1P'][0], CAPS['C2P'][0]
assert (CAPS['C1N'][0], CAPS['C2N'][0]) == (n1, n2), 'branches differ'
C1_MODEL, C2_MODEL = n1 * c_dev['C1P'], n2 * c_dev['C2P']
C1_REAL,  C2_REAL  = n1 * C_UNIT_REAL, n2 * C_UNIT_REAL
print(f'units: C1 {n1}, C2 {n2}, C1/C2 = {n1/n2:.4f}, a1 = 2*C1/C2 = {2*n1/n2:.4f}')
print(f'C1 {C1_MODEL*1e15:.2f} fF model, {C1_REAL*1e15:.0f} fF real, C2 {C2_MODEL*1e15:.1f} fF model, {C2_REAL*1e15:.0f} fF real')
print(f'derived units at {C_UNIT_REAL*1e15:.0f} fF: C1 {C1/C_UNIT_REAL:.1f}, C2 {C2/C_UNIT_REAL:.1f}')
print(f'kT/C1 in band with the drawn C1: {np.sqrt(KB*TEMP/(C1_REAL*OSR))*1e6:.2f} uVrms (real), {np.sqrt(KB*TEMP/(C1_MODEL*OSR))*1e6:.2f} uVrms (model)')

units: C1 2, C2 12, C1/C2 = 0.1667, a1 = 2*C1/C2 = 0.3333
C1 87.95 fF model, 64 fF real, C2 527.7 fF model, 384 fF real
derived units at 32 fF: C1 2.0, C2 12.0
kT/C1 in band with the drawn C1: 11.24 uVrms (real), 9.59 uVrms (model)


### Swing against the OTA output range

With the DAC at `vdd` and `vss` and a full scale of 0.75 V, the state of a first order loop stays inside `a1 * (1 + |u|) * Vref` for an input `u * Vref`. The OTA script gets 0.491 V peak from a simulation of the ideal modulator at -0.5 dBFS with the noise budget added. The OTA output range is the differential peak to peak of its stack at the drawn point.

In [7]:
u_fs = 10 ** (-0.5 / 20)
x_bound = A1 * (1 + u_fs) * V_REF
x_sim = OTA['x_swing']
print(f'state bound at -0.5 dBFS {x_bound:.3f} V peak, ideal modulator simulation {x_sim:.3f} V peak')
print(f'differential peak to peak {2*x_sim:.3f} V against the OTA range {SWING_OTA:.3f} V (model), margin {20*np.log10(SWING_OTA/(2*x_sim)):.1f} dB')
print(f'one step of the state at full scale: {A1*V_REF*2:.3f} V differential, {A1*V_REF:.3f} V per side')

state bound at -0.5 dBFS 0.486 V peak, ideal modulator simulation 0.491 V peak
differential peak to peak 0.983 V against the OTA range 1.604 V (model), margin 4.3 dB
one step of the state at full scale: 0.500 V differential, 0.250 V per side


### OTA settling against the phases

The integrating phase is half a period, 488.3 ns, and the output only moves in that phase. The class budget splits it into slewing and linear settling and asks for `ln(10^(SQNR/20))` time constants. The loop tau comes from the OTA script. It is evaluated again with the model capacitors of the schematic, since the simulations use those and they are 1.37 times the real ones.

The slewing check uses the largest step, `a1 * 2 * Vref` differential, half per side, against the current the folded cascode can put into one output, `Id1`. The load per side is the class `capltot = fanout * C1 + C2 * (1 - beta)`.

In [8]:
def loop(c1, c2):
    beta = c2 / (c1 + c2 + CGG1)
    cl = FANOUT * c1 + c2 * (1 - beta)
    return beta, cl, cl / (GM1 * beta)

n_tau = OTA['n_slin']
for name, c1, c2 in (('real', C1_REAL, C2_REAL), ('model', C1_MODEL, C2_MODEL)):
    beta, cl, tau = loop(c1, c2)
    step = A1 * V_REF                           # per side, largest step
    sr = I_SLEW / cl / 1e6                  # V/us
    v_lin = I_SLEW * tau / cl                   # step size above which the OTA slews
    t_slew = max(step - v_lin, 0) * cl / I_SLEW
    err = min(step, v_lin) * np.exp(-(TS2 - t_slew) / tau) / step
    print(f'{name:5s} caps: beta {beta:.4f}, CL {cl*1e15:.0f} fF, tau {tau*1e9:.2f} ns (limit {OTA["tau"]*1e9:.2f} ns)')
    print(f'      slew rate {sr:.2f} V/us, step {step:.3f} V per side, linear above {v_lin:.3f} V, slewing {t_slew*1e9:.0f} ns')
    print(f'      error at the end of the phase {err:.1e}, needed {10**(-OTA["sqnr_peak_spec"]/20):.1e} (SQNR {OTA["sqnr_peak_spec"]:.1f} dB), time constants in a phase {TS2/tau:.1f} (need {n_tau:.0f} after slewing)')

real  caps: beta 0.7861, CL 210 fF, tau 17.65 ns (limit 30.52 ns)
      slew rate 2.86 V/us, step 0.250 V per side, linear above 0.050 V, slewing 70 ns
      error at the end of the phase 1.0e-11, needed 5.4e-04 (SQNR 65.3 dB), time constants in a phase 27.7 (need 8 after slewing)
model caps: beta 0.8042, CL 279 fF, tau 22.92 ns (limit 30.52 ns)
      slew rate 2.15 V/us, step 0.250 V per side, linear above 0.049 V, slewing 93 ns
      error at the end of the phase 6.5e-09, needed 5.4e-04 (SQNR 65.3 dB), time constants in a phase 21.3 (need 8 after slewing)


### Switch Ron against the same budget

The paths are two switches in series with C1 in phase 1 (`XS_TI`, `XS_TRC`) and three in phase 2 (a DAC rail switch pair and `XS_TRV`). The settling tau of a path is its Ron times C1, `n` of them have to fit in one phase. `Ron` of the transmission gate is read the same way as in `size_sw_tgate_cmfb2.ipynb`, `1/GDS` at a small VDS, once over the potential sweep and once at the points where the rails and `vcm` sit.

In [9]:
nmos = lk(str(DATA / 'sg13cmos5l_lv_nmos.mat'))
pmos = lk(str(DATA / 'sg13cmos5l_lv_pmos.mat'))
VDS = 0.01

def ron_tg(vs):
    vs = np.atleast_1d(np.asarray(vs, dtype=float))
    gn = np.array([nmos.lookup('GDS', L=L_SW, VGS=VDD - v, VDS=VDS, VSB=v) for v in vs]).ravel()
    gp = np.array([pmos.lookup('GDS', L=L_SW, VGS=v, VDS=VDS, VSB=VDD - v) for v in vs]).ravel()
    gn, gp = np.maximum(gn, 0) / nmos['W'], np.maximum(gp, 0) / pmos['W']
    return 1 / (WN * gn + WP * gp)

VS = np.arange(0.3, 1.21, 0.01)         # VSB of both devices stays inside the 1.2 V of the tables
ron = ron_tg(VS)
k = int(np.argmax(ron))
r_vcm = float(ron_tg(VDD / 2)[0])
r_rail = float(max(1 / (WN * nmos.lookup('GDS', L=L_SW, VGS=VDD, VDS=VDS, VSB=0.0) / nmos['W']),
                   1 / (WP * pmos.lookup('GDS', L=L_SW, VGS=VDD, VDS=VDS, VSB=0.0) / pmos['W'])))
print(f'tgate {WN}u/{L_SW}u + {WP}u/{L_SW}u: worst {ron[k]:.0f} ohm at Vs = {VS[k]:.2f} V (sweep 0.3 to 1.2 V), {r_vcm:.0f} ohm at vcm')
print(f'one device alone at its rail: {r_rail:.0f} ohm (the other device is off there)')
R1 = 2 * ron[k]
R2 = 2 * r_rail + ron[k]
for name, c1 in (('real', C1_REAL), ('model', C1_MODEL)):
    print(f'{name:5s} C1: phase 1 tau {R1*c1*1e9:.2f} ns, phase 2 tau {R2*c1*1e9:.2f} ns, {TS2/(R2*c1):.0f} time constants in a phase, Ron allowed for {n_tau:.0f} of them: {TS2/(n_tau*3*c1)/1e3:.0f} kohm per switch')

tgate 2.0u/1.0u + 4.0u/1.0u: worst 4044 ohm at Vs = 1.13 V (sweep 0.3 to 1.2 V), 2524 ohm at vcm
one device alone at its rail: 2945 ohm (the other device is off there)
real  C1: phase 1 tau 0.52 ns, phase 2 tau 0.64 ns, 768 time constants in a phase, Ron allowed for 8 of them: 318 kohm per switch
model C1: phase 1 tau 0.71 ns, phase 2 tau 0.87 ns, 559 time constants in a phase, Ron allowed for 8 of them: 231 kohm per switch


Against the OTA tau of 17.65 ns the switch taus of 0.5 to 0.9 ns are small, so the settling of the integrator is the OTA's. The switch width is not set by Ron. It is set by what the off device leaks on the floating virtual ground, which is next.

### Leakage on the virtual ground

Two things take charge off C2 each cycle. One is the finite OTA gain, which leaves a residual at the virtual ground and keeps part of the old sample: from the charge balance on that node, `p = (C2 + Cp/(1+A)) / (C2 + (C1+Cp)/(1+A))`, so the loss per cycle is about `C1 / (C2 * A0)` and the input capacitance `Cp` drops out to first order. The other is the off tgate on the node, `XS_TRV`, off in phase 1.

The first is checked by iterating the node balance directly, then evaluated with the model and the simulated OTA gain. For the second the tables only start at `VGS = 0`, but the off NMOS sits at `VGS = -vcm` and the off PMOS at `+vcm`. So the subthreshold slope is fitted over `VGS = 0` to 0.15 V and extended to the real gate drive, which is an extrapolation of the model past its table.

In [10]:
def keep(a, c1, c2, cp):
    # fraction of the old C2 state kept per cycle, from the charge balance on the virtual ground
    return (c2 * (1 + a) + cp) / (c1 + c2 * (1 + a) + cp)

A_SIM = 10 ** (58.8 / 20)       # README of ota_fcasc2, simulated
leak_gain = {}
for tag, a in (('model A0', A0_OTA), ('sim A0', A_SIM)):
    leak_gain[tag] = 1 - keep(a, C1_MODEL, C2_MODEL, CGG1)
    print(f'{tag:9s} {20*np.log10(a):.1f} dB: loss per cycle {leak_gain[tag]:.2e}, C1/(C2*A0) = {C1_MODEL/(C2_MODEL*a):.2e}')
print('simulated leak per cycle on the leak bench: 3.50e-4 (ideal capacitors 2.10e-4)')
print(f'integrator gain 1/(1-p) with the model A0: {20*np.log10(1/leak_gain["model A0"]):.1f} dB against the {A0_INT_DB:.0f} dB wanted')

model A0  55.3 dB: loss per cycle 2.86e-04, C1/(C2*A0) = 2.87e-04
sim A0    58.8 dB: loss per cycle 1.91e-04, C1/(C2*A0) = 1.91e-04
simulated leak per cycle on the leak bench: 3.50e-4 (ideal capacitors 2.10e-4)
integrator gain 1/(1-p) with the model A0: 70.9 dB against the 50 dB wanted


In [11]:
# Off tgate on the virtual ground: extend the subthreshold slope of the tables to the real gate drive
def g_off(dev, w, l, vgs_off):
    vg = np.arange(0.0, 0.16, 0.025)
    i = np.array([dev.lookup('ID', L=l, VGS=v, VDS=0.025, VSB=0.0) for v in vg]).ravel() / dev['W']
    slope, ln_i0 = np.polyfit(vg, np.log(i), 1)
    return w * np.exp(ln_i0 + slope * vgs_off) / 0.025, 1 / slope * 1e3     # S at VDS = 25 mV, mV per e-fold

print('off conductance at VDS 25 mV, gate at the rail and the node at vcm, times Ts / C2 = loss per cycle')
for l in (0.13, 0.5, L_SW):
    gn, sn = g_off(nmos, WN, l, -VDD / 2)
    gp, sp = g_off(pmos, WP, l, -VDD / 2)
    print(f'L {l:4.2f} um: nmos {gn:.1e} S, pmos {gp:.1e} S, loss {(gn+gp)/FS/C2_MODEL:.1e} per cycle (slopes {sn:.0f} and {sp:.0f} mV per e-fold)')

off conductance at VDS 25 mV, gate at the rail and the node at vcm, times Ts / C2 = loss per cycle
L 0.13 um: nmos 7.3e-19 S, pmos 2.8e-18 S, loss 6.5e-12 per cycle (slopes 37 and 38 mV per e-fold)
L 0.50 um: nmos 7.7e-19 S, pmos 3.4e-20 S, loss 1.5e-12 per cycle (slopes 33 and 33 mV per e-fold)
L 1.00 um: nmos 7.4e-19 S, pmos 1.2e-20 S, loss 1.4e-12 per cycle (slopes 32 and 32 mV per e-fold)


The finite gain of the OTA accounts for 2.9e-4 of the simulated 3.5e-4 per cycle with the model gain and 1.9e-4 with the simulated 58.8 dB. The extrapolated channel leak of the off tgate is 1.4e-12 per cycle at `L = 1 um` and 6.5e-12 at minimum length, eight orders below, so by the tables the switch channel does not leak at all and the length argument of the README (minimum length is far too leaky) is not reproduced. The extrapolation runs past the table by 0.75 V, so it carries no weight against a simulation. What it does say is that the gap to the simulated total (0.6e-4 with the model gain, 1.6e-4 with the simulated one) is not subthreshold conduction of the channel. Junction and gate leakage are not in the tables.

### Recovery after reset

While `di_rst_b` is low the quantizer holds `do_q` at 0, so the DAC sits at `vss` and every cycle adds `a1 * Vref * (u + 1)` to the state. `sc_int2` has no reset, the OTA runs into its output limit (vod 1.41 V in the sim) and keeps being driven. Two things then set the recovery time, and the OTA slew rate is not one of them.

1. With the output stuck the virtual ground is a passive node. Each cycle C1 shares its charge with C2 and the node moves by `k = C1 / (C1 + C2)` of the gap to the level the DAC and the input ask for, `vcm + Vref * (v - u)`. The charge that piles up on C2 is the output (limited) plus the displacement of both virtual grounds, so the true state is far beyond 1.41 V.
2. After release `do_q` is 1, the DAC reverses and the node moves back at the same `k`. Only when the displacement is gone does the OTA come out of its limit. From there the loop is the ideal integrator again and the state falls by `a1 * Vref * (1 - u)` per clock, which goes to zero as the input approaches full scale. That is why the time grows with `vi`.

A behavioural model of both stages with the drawn `k`, the 1.41 V limit and a reset long enough to reach the end level, counting clocks until the state is back inside its bound `a1 * (1 + u) * Vref`. It leaves out the CMFB, the junction clamps on the virtual ground and the OTA's own recovery time. The README gives 16 to 17 clocks at vi = +0.3 V.

In [12]:
K = C1_MODEL / (C1_MODEL + C2_MODEL)

def recover(u, n_rst=61, k=K, vsat=V_SAT):
    # differential state vc2 = vo - delta, delta is the virtual ground displacement, vo limited to vsat
    vc2, delta, sat = 0.0, 0.0, False
    for n in range(n_rst + 200):
        v = -1.0 if n < n_rst else 1.0
        if n == n_rst:
            vc2_rel, d_rel = vc2, delta
        if not sat:
            vc2 += A1 * V_REF * (u - v)
            if abs(vc2) > vsat:
                sat, delta = True, np.sign(vc2) * vsat - vc2
        else:
            delta += k * (2 * V_REF * (v - u) - delta)
            vc2 = np.sign(vc2) * vsat - delta
            if abs(vc2) <= vsat:
                sat, delta = False, 0.0
        if n >= n_rst and abs(vc2) <= A1 * (1 + u) * V_REF:
            return n - n_rst + 1, abs(vc2_rel), d_rel
    return None

print('vi (V)   u     state at release (V)   clocks to recover   virtual ground at release (V from vcm)')
for vi in (0.0, 0.3, 0.5, 0.6, 0.7):
    u = vi / V_REF
    n, x_rel, d_rel = recover(u)
    print(f'{vi:5.2f}  {u:5.2f}   {x_rel:6.2f}                 {n:3d}                 {d_rel/2:+.2f} per side')

vi (V)   u     state at release (V)   clocks to recover   virtual ground at release (V from vcm)
 0.00   0.00     2.91                  10                 -0.75 per side
 0.30   0.40     3.51                  15                 -1.05 per side
 0.50   0.67     3.91                  24                 -1.25 per side
 0.60   0.80     4.11                  35                 -1.35 per side
 0.70   0.93     4.31                  78                 -1.45 per side


In [13]:
# Where the clocks go at vi = +0.3 V, and the OTA slewing for comparison
u = 0.3 / V_REF
gap = 2 * V_REF * (1 + u)
n_pass = np.log(1 - (V_REF * (1 + u)) / (V_REF * (1 + u) + V_REF * (1 - u))) / np.log(1 - K)
n_ramp = (V_SAT - A1 * (1 + u) * V_REF) / (A1 * V_REF * (1 - u))
_, cl, tau = loop(C1_MODEL, C2_MODEL)
print(f'passive stage: the node closes {K:.3f} of the gap per cycle, about {n_pass:.1f} clocks to get the OTA out of its limit')
print(f'ideal stage: {A1*V_REF*(1-u):.3f} V per clock differential, {n_ramp:.1f} clocks from {V_SAT} V to the bound')
print(f'OTA slewing {I_SLEW/cl/1e6:.2f} V/us per side moves {I_SLEW/cl*TS2:.2f} V per side in a phase, the loop asks for at most {A1*V_REF:.2f} V per side per clock')

passive stage: the node closes 0.143 of the gap per cycle, about 7.8 clocks to get the OTA out of its limit
ideal stage: 0.150 V per clock differential, 7.1 clocks from 1.41 V to the bound
OTA slewing 2.15 V/us per side moves 1.05 V per side in a phase, the loop asks for at most 0.25 V per side per clock


The model gives 15 clocks at vi = +0.3 V (7.8 with the OTA out of its limit, 7.1 on the ramp) against the 16 to 17 of the simulation, and the trend to full scale is the same: 10 clocks at vi = 0, 24 at +0.5 V, 78 at +0.7 V. Recovery is limited by the charge sharing ratio `C1/(C1+C2) = 1/7` while the OTA is out, and by the `a1 * (1 - u)` drive of the loop afterwards. The OTA slew rate is 2.1 V/us per side with the model capacitors, which moves 1.05 V per side in a phase against the 0.25 V the loop asks for, so it does not matter. The model leaves the displacement of the virtual grounds unclamped. At vi = 0.3 V it is -1.05 V per side, which puts the node at -0.30 V, below `vss`. A junction clamp there would shorten the first stage, yet the simulation is slower than the model, so the missing 1 to 2 clocks come from something this model leaves out, most likely the OTA and the CMFB coming back from the limit.

For a later fix, the knobs are in this model: a reset switch across C2 or on the virtual grounds removes both stages, a lower C1/C2 only makes the passive stage slower, and a faster loop (larger a1) shortens the ramp but moves the whole design. None of these is done here.

### Derived against drawn

The table checks every size and every figure of merit this notebook derives against the schematic or the README simulation.

In [14]:
n_model = recover(0.3 / V_REF)[0]
rows = [
    # name, derived, drawn, unit, ok
    ('C1 units of 32 fF (real)', C1 / C_UNIT_REAL, n1, '', abs(C1 / C_UNIT_REAL - n1) < 0.01),
    ('C2 units of 32 fF (real)', C2 / C_UNIT_REAL, n2, '', abs(C2 / C_UNIT_REAL - n2) < 0.01),
    ('C1 / C2', C1 / C2, C1_MODEL / C2_MODEL, '', abs(C1 / C2 - C1_MODEL / C2_MODEL) < 1e-6),
    ('C1 absolute (schematic holds the model value)', C1 * 1e15, C1_MODEL * 1e15, 'fF', False),
    ('kT/C1 in band, drawn model C1', noise_c1 * 1e6, np.sqrt(KB * TEMP / (C1_MODEL * OSR)) * 1e6, 'uVrms', True),
    ('state swing, differential peak to peak', 2 * x_sim, SWING_OTA, 'V', 2 * x_sim < SWING_OTA),
    ('loop tau with model caps (limit)', OTA['tau'] * 1e9, loop(C1_MODEL, C2_MODEL)[2] * 1e9, 'ns', loop(C1_MODEL, C2_MODEL)[2] < OTA['tau']),
    ('Ron allowed per switch (derived) / worst drawn', TS2 / (n_tau * 3 * C1_MODEL) / 1e3, ron[k] / 1e3, 'kohm', ron[k] < TS2 / (n_tau * 3 * C1_MODEL)),
    ('leak per cycle, finite gain (model A0) / sim', leak_gain['model A0'] * 1e4, 3.5, 'e-4', abs(leak_gain['model A0'] * 1e4 - 3.5) / 3.5 < 0.25),
    ('reset recovery at +0.3 V / sim 16 to 17', n_model, 16.5, 'clocks', abs(n_model - 16.5) <= 2),
]
print(f'{"quantity":48s}{"derived":>10s}{"drawn/sim":>11s}  unit    agree')
for name, d, s, unit, ok in rows:
    print(f'{name:48s}{d:10.3g}{s:11.3g}  {unit:7s} {"yes" if ok else "NO"}')

quantity                                           derived  drawn/sim  unit    agree
C1 units of 32 fF (real)                                 2          2          yes
C2 units of 32 fF (real)                                12         12          yes
C1 / C2                                              0.167      0.167          yes
C1 absolute (schematic holds the model value)           64         88  fF      NO
kT/C1 in band, drawn model C1                         11.2       9.59  uVrms   yes
state swing, differential peak to peak               0.983        1.6  V       yes
loop tau with model caps (limit)                      30.5       22.9  ns      yes
Ron allowed per switch (derived) / worst drawn         231       4.04  kohm    yes
leak per cycle, finite gain (model A0) / sim          2.86        3.5  e-4     yes
reset recovery at +0.3 V / sim 16 to 17                 15       16.5  clocks  yes


### Result

The drawn sizes follow from the spec: C1 and C2 are 2 and 12 units, the ratio is exactly the 1/6 of the class design, the swing fits the OTA with 4.3 dB to spare, the OTA tau is 17.65 ns against 30.5 ns allowed (22.9 ns with the model capacitors), and the switch Ron adds under 1 ns of tau. The disagreements, all known, are listed here and not changed.

- The schematic holds the model unit of 43.98 fF, so every capacitor is 1.37 times the 32 fF unit the derivation assumes. Ratios and the leak are unaffected, the time constants and the slewing are 1.37 times slower in simulation than in silicon.
- The leak from the finite gain alone is 2.9e-4 per cycle with the model gain, the simulated total is 3.5e-4. The remainder (0.6e-4) is not separated out, and the tables give no channel leak of the off switch at any L, unlike the README.
- The reset recovery model gives 15 clocks, the simulation 16 to 17. The model has no clamp on the virtual ground and no OTA or CMFB recovery time.
- kT/C asks for 0.1 fF. The 64 fF are set by leakage and droop, so the kT/C line is no sizing argument for this block.